<a href="https://colab.research.google.com/github/DinethPerera-eng/Statistical-Learning-e22285/blob/main/Assignment_7D_E22285.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 7D: Structural Health Monitoring

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import beta as beta_distribution

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

## 1. Prior Belief Boundaries

The initial belief about the remaining stiffness efficiency factor is

$$
\Theta\sim\operatorname{Beta}(8,1.5).
$$

The Beta probability density function is

$$
f_{\Theta}^{(0)}(\theta)
=
\frac{1}{B(8,1.5)}
\theta^{8-1}(1-\theta)^{1.5-1},
\qquad 0<\theta<1.
$$

The expected value of a Beta distribution is

$$
E[\Theta]
=
\frac{\alpha}{\alpha+\beta}.
$$

Therefore,

$$
E[\Theta^{(0)}]
=
\frac{8}{8+1.5}
=
\frac{8}{9.5}
\approx 0.8421.
$$

The prior mean indicates that the component is expected to retain approximately 84.21% of its original stiffness before sensor measurements are collected. The distribution places most of its probability mass close to the healthy end of the physical range because $\alpha$ is much larger than $\beta$.

This is an appropriate initial engineering prior because a component that has passed manufacturing inspection and entered service without a known impact event is normally expected to be mostly healthy. However, the distribution still assigns nonzero probability to lower stiffness values. Therefore, the model remains capable of learning that degradation has occurred when the sensor measurements provide sufficient evidence.

In [ ]:
theta_grid_prior = np.linspace(0.01, 1.0, 1500)

prior_density = beta_distribution.pdf(
    theta_grid_prior,
    8,
    1.5,
)

prior_density = prior_density / np.trapezoid(
    prior_density,
    theta_grid_prior,
)

prior_mean_analytical = 8 / (8 + 1.5)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=theta_grid_prior,
        y=prior_density,
        mode="lines",
        name="Beta(8, 1.5) prior",
    )
)

fig.add_vline(
    x=prior_mean_analytical,
    line_dash="dash",
    annotation_text=f"Prior mean = {prior_mean_analytical:.4f}",
)

fig.update_layout(
    title="Initial Prior for Remaining Stiffness Efficiency",
    xaxis_title="Remaining stiffness efficiency, theta",
    yaxis_title="Probability density",
    template="plotly_white",
    width=900,
    height=520,
)

fig.show()

print(f"Expected prior stiffness efficiency: {prior_mean_analytical:.4f}")

Expected prior stiffness efficiency: 0.8421


## 2. Structural Likelihood Formulation

The structural measurement model is

$$
Y_k
=
\theta K_{\text{nominal}}e^{\epsilon_k},
\qquad
\epsilon_k\sim N(0,\sigma^2).
$$

Taking the natural logarithm of both sides gives

$$
\log Y_k
=
\log(\theta K_{\text{nominal}})
+
\epsilon_k.
$$

Because $\epsilon_k$ is normally distributed,

$$
\log Y_k\mid\Theta=\theta
\sim
N\left(
\log(\theta K_{\text{nominal}}),
\sigma^2
\right).
$$

Therefore,

$$
Y_k\mid\Theta=\theta
\sim
\operatorname{LogNormal}
\left(
\log(\theta K_{\text{nominal}}),
\sigma^2
\right).
$$

For a single positive sensor measurement $y_k$, the likelihood contribution is

$$
L(y_k\mid\theta)
=
\frac{1}{y_k\sigma\sqrt{2\pi}}
\exp\left[
-\frac{
\left(
\log y_k
-
\log(\theta K_{\text{nominal}})
\right)^2
}{
2\sigma^2
}
\right],
\qquad
y_k>0.
$$

An equivalent expression is

$$
L(y_k\mid\theta)
=
\frac{1}{y_k\sigma\sqrt{2\pi}}
\exp\left[
-\frac{
\left(
\log\frac{y_k}{\theta K_{\text{nominal}}}
\right)^2
}{
2\sigma^2
}
\right].
$$

Assuming the sensor readings are conditionally independent given $\theta$, the joint likelihood of the running history

$$
\mathbf y^{(k)}
=
(y_1,y_2,\ldots,y_k)
$$

is

$$
L(\mathbf y^{(k)}\mid\theta)
=
\prod_{i=1}^{k}
L(y_i\mid\theta).
$$

Thus,

$$
L(\mathbf y^{(k)}\mid\theta)
=
\prod_{i=1}^{k}
\frac{1}{y_i\sigma\sqrt{2\pi}}
\exp\left[
-\frac{
\left(
\log\frac{y_i}{\theta K_{\text{nominal}}}
\right)^2
}{
2\sigma^2
}
\right].
$$

The log-normal model is suitable because it guarantees positive stiffness measurements. It also represents multiplicative sensor variation, where the size of the error changes proportionally with the measured stiffness.

## 3. Mathematical Formulation of the Non-Conjugate Grid Update

The Beta prior has the kernel

$$
f_{\Theta}^{(0)}(\theta)
\propto
\theta^{\alpha-1}(1-\theta)^{\beta-1}.
$$

The log-normal likelihood contains the term

$$
\exp\left[
-\frac{
\left(
\log y_k
-
\log(\theta K_{\text{nominal}})
\right)^2
}{
2\sigma^2
}
\right].
$$

This likelihood cannot be combined with the Beta prior by simply changing the two Beta shape parameters. In particular, the squared logarithmic term involving $\log\theta$ does not have the same algebraic structure as the Beta kernel. Therefore, the Beta distribution is not conjugate to this log-normal likelihood.

Let

$$
f_{k-1}(\theta)
=
f_{\Theta\mid\mathbf Y^{(k-1)}}
\left(
\theta\mid\mathbf y^{(k-1)}
\right)
$$

be the posterior after the first $k-1$ sensor measurements. When the new measurement $y_k$ is received, the recursive Bayesian update is

$$
f_k(\theta)
\propto
L(y_k\mid\theta)
f_{k-1}(\theta),
\qquad
0<\theta\le1.
$$

Substituting the log-normal likelihood gives

$$
f_k(\theta)
\propto
\frac{1}{y_k\sigma\sqrt{2\pi}}
\exp\left[
-\frac{
\left(
\log\frac{y_k}{\theta K_{\text{nominal}}}
\right)^2
}{
2\sigma^2
}
\right]
f_{k-1}(\theta).
$$

The normalized posterior is

$$
f_k(\theta)
=
\frac{
L(y_k\mid\theta)f_{k-1}(\theta)
}{
\int_{0}^{1}
L(y_k\mid u)f_{k-1}(u)\,du
}.
$$

Because the normalizing integral has no convenient closed-form expression, the posterior must be evaluated numerically. A bounded grid is suitable because the stiffness factor has the known physical range $(0,1]$.

## 4. Running Point Estimates

The running posterior mean is the Bayesian point estimate under squared-error loss. It is defined by

$$
\widehat{\theta}_{\mathrm{Bayes}}^{(k)}
=
E[\Theta\mid\mathbf Y^{(k)}=\mathbf y^{(k)}].
$$

Over the bounded physical domain, it is

$$
\boxed{
\widehat{\theta}_{\mathrm{Bayes}}^{(k)}
=
\int_{0}^{1}
\theta
f_k(\theta)\,d\theta
}.
$$

Because the posterior density is normalized,

$$
\int_{0}^{1}f_k(\theta)\,d\theta=1.
$$

If the numerical posterior is not yet normalized, the posterior mean can be written as

$$
\widehat{\theta}_{\mathrm{Bayes}}^{(k)}
=
\frac{
\int_{0}^{1}
\theta\widetilde f_k(\theta)\,d\theta
}{
\int_{0}^{1}
\widetilde f_k(\theta)\,d\theta
}.
$$

The running maximum a posteriori estimate is the value of $\theta$ where the posterior density reaches its highest value:

$$
\boxed{
\widehat{\theta}_{\mathrm{MAP}}^{(k)}
=
\operatorname*{arg\,max}_{0<\theta\le1}
f_k(\theta)
}.
$$

On a discrete grid $\theta_1,\theta_2,\ldots,\theta_m$, the estimators are approximated by

$$
\widehat{\theta}_{\mathrm{Bayes}}^{(k)}
\approx
\operatorname{trapz}
\left(
\theta f_k(\theta),
\theta
\right)
$$

and

$$
\widehat{\theta}_{\mathrm{MAP}}^{(k)}
=
\theta_j
\quad\text{where}\quad
j
=
\operatorname{argmax}
f_k(\theta_j).
$$

The posterior mean uses the complete distribution, while the MAP uses only the location of the highest density. Therefore, the two estimates may differ slightly when the posterior is skewed.

## 5. Algorithmic Grid Approximation and Normalization

A numerical grid can be used to represent the posterior distribution at every inspection step.

1. Select a bounded grid:

$$
\theta_j\in[0.01,1.0].
$$

The lower boundary is set to $0.01$ instead of exactly zero because the likelihood contains $\log\theta$, and $\log 0$ is undefined.

2. Evaluate the initial prior on the grid:

$$
f_0(\theta_j)
=
\operatorname{BetaPDF}
(\theta_j;8,1.5).
$$

3. Normalize the initial prior using the trapezoidal rule:

$$
f_0(\theta_j)
\leftarrow
\frac{
f_0(\theta_j)
}{
\operatorname{trapz}
(f_0,\theta)
}.
$$

4. When a new sensor measurement $y_k$ arrives, calculate the likelihood at every grid point:

$$
L_k(\theta_j)
=
\frac{1}{y_k\sigma\sqrt{2\pi}}
\exp\left[
-\frac{
\left(
\log\frac{y_k}{\theta_jK_{\text{nominal}}}
\right)^2
}{
2\sigma^2
}
\right].
$$

5. Multiply the previous posterior by the new likelihood:

$$
\widetilde f_k(\theta_j)
=
L_k(\theta_j)
f_{k-1}(\theta_j).
$$

6. Calculate the numerical normalizing constant:

$$
Z_k
\approx
\operatorname{trapz}
\left(
\widetilde f_k,
\theta
\right).
$$

7. Normalize the updated posterior:

$$
f_k(\theta_j)
=
\frac{
\widetilde f_k(\theta_j)
}{
Z_k
}.
$$

8. Calculate the posterior mean:

$$
\widehat{\theta}_{\mathrm{Bayes}}^{(k)}
\approx
\operatorname{trapz}
\left(
\theta f_k(\theta),
\theta
\right).
$$

9. Calculate the MAP estimate by finding the grid point with the largest posterior density.

10. Store selected posterior curves and point estimates for later visualization.

A sufficiently fine grid improves the numerical approximation. The grid should also cover the complete physically possible region so that important posterior probability is not removed by the computational boundaries.

## 6. Performance Tracking and Degradation Convergence Analysis

The true remaining stiffness after the impact is

$$
\theta_{\text{true}}=0.68.
$$

The nominal stiffness and log-space sensor noise are

$$
K_{\text{nominal}}=50.0\text{ kN/mm}
$$

and

$$
\sigma=0.15.
$$

The sensor stream is simulated using

$$
y_k
=
\theta_{\text{true}}
K_{\text{nominal}}
e^{\epsilon_k},
\qquad
\epsilon_k\sim N(0,\sigma^2).
$$

The posterior is updated sequentially for 15 sensor measurements. Posterior density curves are stored at

$$
k\in\{0,1,2,5,10,15\}.
$$

In [ ]:
rng = np.random.default_rng(42)

theta_true = 0.68
K_nominal = 50.0
sigma = 0.15
number_of_measurements = 15

theta_grid = np.linspace(0.01, 1.0, 2000)

posterior = beta_distribution.pdf(
    theta_grid,
    8,
    1.5,
)

posterior = posterior / np.trapezoid(
    posterior,
    theta_grid,
)

posterior_mean_initial = np.trapezoid(
    theta_grid * posterior,
    theta_grid,
)

posterior_map_initial = theta_grid[
    np.argmax(posterior)
]

posterior_variance_initial = np.trapezoid(
    (theta_grid - posterior_mean_initial) ** 2
    * posterior,
    theta_grid,
)

posterior_sd_initial = np.sqrt(
    posterior_variance_initial
)

posterior_mean_history = [
    posterior_mean_initial
]

posterior_map_history = [
    posterior_map_initial
]

posterior_sd_history = [
    posterior_sd_initial
]

milestones = {0, 1, 2, 5, 10, 15}

posterior_curves = {
    0: posterior.copy()
}

measurement_records = []

for step in range(
    1,
    number_of_measurements + 1,
):
    epsilon_k = rng.normal(
        0,
        sigma,
    )

    sensor_reading = (
        theta_true
        * K_nominal
        * np.exp(epsilon_k)
    )

    log_ratio = np.log(
        sensor_reading
        / (theta_grid * K_nominal)
    )

    likelihood = (
        1
        / (
            sensor_reading
            * sigma
            * np.sqrt(2 * np.pi)
        )
        * np.exp(
            -(log_ratio ** 2)
            / (2 * sigma ** 2)
        )
    )

    unnormalized_posterior = (
        posterior * likelihood
    )

    normalizing_constant = np.trapezoid(
        unnormalized_posterior,
        theta_grid,
    )

    posterior = (
        unnormalized_posterior
        / normalizing_constant
    )

    posterior_mean = np.trapezoid(
        theta_grid * posterior,
        theta_grid,
    )

    posterior_map = theta_grid[
        np.argmax(posterior)
    ]

    posterior_variance = np.trapezoid(
        (theta_grid - posterior_mean) ** 2
        * posterior,
        theta_grid,
    )

    posterior_sd = np.sqrt(
        posterior_variance
    )

    posterior_mean_history.append(
        posterior_mean
    )

    posterior_map_history.append(
        posterior_map
    )

    posterior_sd_history.append(
        posterior_sd
    )

    if step in milestones:
        posterior_curves[step] = (
            posterior.copy()
        )

    measurement_records.append(
        {
            "Step": step,
            "Sensor reading": sensor_reading,
            "Posterior mean": posterior_mean,
            "MAP": posterior_map,
            "Posterior SD": posterior_sd,
            "Mean error": abs(
                posterior_mean - theta_true
            ),
        }
    )

measurement_results = pd.DataFrame(
    measurement_records
)

fig_density = go.Figure()

for milestone in sorted(
    posterior_curves.keys()
):
    fig_density.add_trace(
        go.Scatter(
            x=theta_grid,
            y=posterior_curves[milestone],
            mode="lines",
            name=f"k = {milestone}",
        )
    )

fig_density.add_vline(
    x=theta_true,
    line_dash="dash",
    annotation_text="True stiffness = 0.68",
)

fig_density.update_layout(
    title="Sequential Posterior Density Curves",
    xaxis_title="Remaining stiffness efficiency, theta",
    yaxis_title="Posterior density",
    template="plotly_white",
    width=950,
    height=550,
)

fig_density.show()

steps = np.arange(
    0,
    number_of_measurements + 1,
)

fig_estimates = go.Figure()

fig_estimates.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_mean_history,
        mode="lines+markers",
        name="Posterior mean",
    )
)

fig_estimates.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_map_history,
        mode="lines+markers",
        name="MAP estimate",
    )
)

fig_estimates.add_hline(
    y=theta_true,
    line_dash="dash",
    annotation_text="True stiffness = 0.68",
)

fig_estimates.update_layout(
    title="Convergence of Structural Stiffness Estimates",
    xaxis_title="Number of sensor measurements",
    yaxis_title="Estimated remaining stiffness",
    template="plotly_white",
    width=950,
    height=540,
)

fig_estimates.show()

confidence_steps = [
    row["Step"]
    for row in measurement_records
    if abs(
        row["Posterior mean"]
        - theta_true
    ) <= 0.03
    and row["Posterior SD"] <= 0.05
]

first_confident_step = (
    confidence_steps[0]
    if confidence_steps
    else None
)

print(
    f"Initial posterior mean: "
    f"{posterior_mean_history[0]:.4f}"
)

print(
    f"Final posterior mean: "
    f"{posterior_mean_history[-1]:.4f}"
)

print(
    f"Final MAP estimate: "
    f"{posterior_map_history[-1]:.4f}"
)

print(
    f"Initial posterior SD: "
    f"{posterior_sd_history[0]:.4f}"
)

print(
    f"Final posterior SD: "
    f"{posterior_sd_history[-1]:.4f}"
)

print(
    f"First confident isolation step: "
    f"{first_confident_step}"
)

display(measurement_results)

Initial posterior mean: 0.8421
Final posterior mean: 0.6873
Final MAP estimate: 0.6860
Initial posterior SD: 0.1125
Final posterior SD: 0.0266
First confident isolation step: 5


,Step,Sensor reading,Posterior mean,MAP,Posterior SD,Mean error
0,1,35.590121,0.794674,0.796948,0.092630,0.114674
1,2,29.089082,0.697669,0.687499,0.071890,0.017669
2,3,38.051032,0.717735,0.710775,0.060896,0.037735
3,4,39.151754,0.733173,0.727614,0.054073,0.053173
4,5,25.373498,0.682290,0.678089,0.045434,0.002290
5,6,27.967234,0.660357,0.656793,0.040243,0.019643
6,7,34.658277,0.664911,0.661746,0.037536,0.015089
7,8,32.424819,0.662861,0.660260,0.035026,0.017139
8,9,33.914422,0.664548,0.662241,0.033119,0.015452
9,10,29.916314,0.657658,0.655308,0.031110,0.022342


### Analysis

The initial prior is optimistic because its mean is approximately $0.8421$. Therefore, before receiving measurements, the model expects the structure to retain more than 84% of its nominal stiffness. The true post-impact value of $0.68$ is substantially lower than this initial belief.

The first sensor readings produce a large downward movement in both the posterior mean and the MAP estimate. This happens because measurements generated around

$$
0.68\times50
=
34.0\text{ kN/mm}
$$

are more compatible with stiffness values near $0.68$ than with values near the original prior mean.

To state when the system has confidently isolated the damage state, the simulation uses the following numerical criterion:

$$
\left|
\widehat{\theta}_{\mathrm{Bayes}}^{(k)}
-
0.68
\right|
\le0.03
$$

and

$$
\operatorname{SD}
\left(
\Theta\mid\mathbf y^{(k)}
\right)
\le0.05.
$$

With the fixed random seed, this condition is first satisfied after approximately five sensor readings. This indicates that the accumulated likelihood from the sensor stream is strong enough to overcome the initially healthy prior after a relatively small number of inspections.

The posterior curves become progressively narrower as additional measurements are processed. The narrowing means that the range of stiffness values considered plausible by the model is becoming smaller. In structural health monitoring, this reduction in uncertainty is important because safety actions are often based on thresholds.

For example, if an inspection threshold were defined near $\theta=0.70$, an initially wide posterior could place meaningful probability on both sides of the threshold. This would make the safety decision uncertain. After several consistent measurements, the posterior becomes concentrated near $0.68$, providing stronger evidence that the structure has fallen below the threshold.

A narrow posterior does not mean the structure is safe. It means the system is more certain about the estimated degradation state. When the concentrated posterior lies below a safety threshold, the evidence for inspection, repair, load restriction, or shutdown becomes stronger. Actual engineering decisions should still account for model assumptions, sensor calibration, environmental effects, and conservative safety margins.